In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

In [4]:
df = pd.read_parquet("../data/interim/train.parquet")

with open("../config/features.json") as file:
    feature_cols = json.load(file)

X = df[feature_cols].copy()
y = df['target'].copy()

In [7]:
# tabela para iniciar as investigações
feature_profile = pd.DataFrame({
    "dtype": X.dtypes,
    "missing_count": X.isna().sum(),
    "missing_pct": X.isna().mean() * 100,
    "unique_values": X.nunique()
})

feature_profile = feature_profile.sort_values(
    "missing_pct",
    ascending = False
)

display(feature_profile)

,dtype,missing_count,missing_pct,unique_values
koi_ingress,float64,6068,100.0,0
koi_sma_err2,float64,6068,100.0,0
koi_sage,float64,6068,100.0,0
koi_eccen_err1,float64,6068,100.0,0
koi_eccen_err2,float64,6068,100.0,0
...,...,...,...,...
koi_period,float64,0,0.0,6068
koi_duration,float64,0,0.0,5443
koi_time0bk,float64,0,0.0,6055
koi_kepmag,float64,0,0.0,3152


In [ ]:
missing_summary = pd.cut(
    feature_profile["missing_pct"],
    bins=[-1, 0, 5, 25, 50, 75, 99, 100],
    labels=[
        "0%",
        "0-5%",
        "5-25%",
        "25-50%",
        "50-75%",
        "75-99%",
        "99-100%"
    ]
).value_counts(sort=False)

print("faixa de missing das variáveis elegiveis")
display(missing_summary)

missing_pct
0%          5
0-5%       62
5-25%       5
25-50%      0
50-75%      0
75-99%      0
99-100%    24
Name: count, dtype: int64

In [9]:
all_null_cols = feature_profile.query("missing_pct == 100")
constant_cols = feature_profile.query("unique_values == 1")

print("Completamente nulas:", len(all_null_cols))
print("Constantes entre valores observados:", len(constant_cols))

display(all_null_cols)
display(constant_cols)

Completamente nulas: 24
Constantes entre valores observados: 3


,dtype,missing_count,missing_pct,unique_values
koi_ingress,float64,6068,100.0,0
koi_sma_err2,float64,6068,100.0,0
koi_sage,float64,6068,100.0,0
koi_eccen_err1,float64,6068,100.0,0
koi_eccen_err2,float64,6068,100.0,0
koi_longp,float64,6068,100.0,0
koi_longp_err1,float64,6068,100.0,0
koi_longp_err2,float64,6068,100.0,0
koi_gmag_err,float64,6068,100.0,0
koi_model_dof,float64,6068,100.0,0


,dtype,missing_count,missing_pct,unique_values
koi_eccen,float64,207,3.411338,1
koi_ldm_coeff3,float64,207,3.411338,1
koi_ldm_coeff4,float64,207,3.411338,1


In [ ]:
cols = [
    "koi_eccen",
    "koi_ldm_coeff3",
    "koi_ldm_coeff4"
]

# correlação entre as variáveis constantes
X[cols].isna().astype(int).corr()

,koi_eccen,koi_ldm_coeff3,koi_ldm_coeff4
koi_eccen,1.0,1.0,1.0
koi_ldm_coeff3,1.0,1.0,1.0
koi_ldm_coeff4,1.0,1.0,1.0


In [11]:
missing_by_target = X.isna().groupby(y).mean().T * 100

missing_by_target.columns = [
    "false_positive",
    "confirmed"
]

missing_by_target["difference_pp"] = (
    missing_by_target["false_positive"]
    - missing_by_target["confirmed"]
).abs()

display(
    missing_by_target
    .sort_values("difference_pp", ascending=False)
    .head(15)
    .round(2)
)

,false_positive,confirmed,difference_pp
koi_bin_oedp_sig,18.14,0.64,17.50
koi_max_mult_ev,13.54,0.41,13.13
koi_max_sngle_ev,13.54,0.41,13.13
koi_num_transits,13.54,0.41,13.13
koi_smass_err2,7.29,0.09,7.20
koi_srad_err2,7.29,0.09,7.20
koi_slogg_err2,7.29,0.09,7.20
koi_srad_err1,7.29,0.09,7.20
koi_smass_err1,7.29,0.09,7.20
koi_slogg_err1,7.29,0.09,7.20


In [13]:
display(X[cols].isna().astype(int).corr())

display(missing_by_target.head(3).round(2))

,koi_eccen,koi_ldm_coeff3,koi_ldm_coeff4
koi_eccen,1.0,1.0,1.0
koi_ldm_coeff3,1.0,1.0,1.0
koi_ldm_coeff4,1.0,1.0,1.0


,false_positive,confirmed,difference_pp
koi_gmag,0.78,0.05,0.73
koi_gmag_err,100.00,100.00,0.00
koi_rmag,0.13,0.00,0.13
